<a href="https://colab.research.google.com/github/mdonbruce/AspNetDocs/blob/master/01_instructor_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: Handling User Input Errors Securely — Instructor (Executed)

**Objective:** Safely parse numeric inputs and produce a secure summary.

**Dataset:** `user_inputs_large.csv`

This notebook is a complete reference solution with outputs.

In [2]:
import pandas as pd, numpy as np

df = pd.read_csv('user_inputs_large.csv')
df.head()

,input_value,input_type_hint,source,parseable_numeric
0,-2041,int_str,web_form,True
1,-136.33,float_str,web_form,True
2,-4435,int_str,api_payload,True
3,-2256,int_str,cli,True
4,-2318,int_str,api_payload,True


In [3]:
def safe_parse_number(x):
    if x is None:
        return (False, None)
    s = str(x).strip()
    if s == '' or s.startswith('$'):
        return (False, None)
    try:
        val = float(s.replace(',', ''))
        if not np.isfinite(val):
            return (False, None)
        return (True, float(val))
    except Exception:
        return (False, None)

parsed = df['input_value'].apply(safe_parse_number)
df['is_valid_number'] = parsed.apply(lambda t: t[0])
df['parsed_value'] = parsed.apply(lambda t: t[1])

invalid = int((~df['is_valid_number']).sum())
valid = int(df['is_valid_number'].sum())
print('valid:', valid, 'invalid:', invalid)

valid: 3971 invalid: 1029


In [4]:
df.loc[df['is_valid_number'], 'parsed_value'].astype(float).describe()

,parsed_value
count,3971.000000
mean,-6.243165
std,2639.304732
min,-4999.000000
25%,-1973.500000
50%,15.000000
75%,1904.500000
max,5000.000000


In [5]:
top_sources = (df.loc[~df['is_valid_number']]
               .groupby('source').size()
               .sort_values(ascending=False).head(5))
top_sources

,0
source,
csv_import,267
api_payload,262
web_form,254
cli,246
